In [ ]:
# !pip install -q transformers datasets sentencepiece onnx onnxruntime onnxscript 
# можно: lime

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/zloelias/kinopoisk-reviews/" + splits["train"])

In [ ]:
df = df.drop(columns = ['label_name', '__index_level_0__'])
df

In [ ]:
test = pd.read_parquet("hf://datasets/zloelias/kinopoisk-reviews/" + splits["test"])

In [ ]:
print(f"Кол-во дупликатов: {df['text'].duplicated().sum()}")

#Удалим дубликаты
df_new = df.drop_duplicates(['text'], keep='first')

print(f"Распределение классов у таргета: {df_new['labels'].value_counts()}")

In [ ]:
df_test = test.drop_duplicates(['text'], keep = 'first')

print(f"Распределение классов у таргета в тестовом датасете: {df_test['labels'].value_counts()}")

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(df_new, test_size= .1, stratify= df_new['labels'], random_state= 42)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import warnings
from tqdm.auto import tqdm
import os
import random
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
import seaborn as sns
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

In [ ]:
MODEL_NAME = "intfloat/multilingual-e5-small"
BATCH_SIZE = 32
MAX_LENGTH = 256
NUM_EPOCHS = 6
LR = 3e-5
RANDOM_SEED = 42

In [ ]:
def seed_all(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(RANDOM_SEED)

In [ ]:
#Создадим класс Dataset
class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_length = 128):
        self.texts = df['text'].tolist()
        self.labels = df['labels'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype= torch.float)
        }

In [ ]:
#Создаем все необходимые датасеты
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = TextDataset(df_train, tokenizer, MAX_LENGTH)

val_dataset = TextDataset(df_val, tokenizer, MAX_LENGTH)

test_dataset = TextDataset(df_test, tokenizer, MAX_LENGTH)

In [ ]:
#Создадим даталоадеры
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory = True, num_workers = 2)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory = True, num_workers = 2)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory = True, num_workers = 2)

In [ ]:
class E5Classifier(nn.Module):
    def __init__(self, model_name='intfloat/multilingual-e5-small'):
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name)
        print("embed_dim = ", self.model.config.hidden_size)
        self.classifier = nn.Linear(self.model.config.hidden_size, 1) # бинарная классификация
        
    def mean_pooling(self, token_embeddings, attention_mask):
        """Усреднение векторов всех токенов кроме padding"""
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(
            token_embeddings.size()
        ).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return sum_embeddings / sum_mask
        
    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        
        # Average pooling
        pooled_output = self.mean_pooling(
            outputs.last_hidden_state, 
            attention_mask
        )
        
        logits = self.classifier(pooled_output)
        return logits.squeeze(-1)

In [ ]:
# Инициализируем модель
model = E5Classifier(model_name=MODEL_NAME)

In [ ]:
# Функция для вычисления числа параметров
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

count_parameters(model)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha: torch.Tensor, gamma: float = 2.0, reduction: str = "mean"):
        """
        Focal Loss для бинарной классификации.
        
        Parameters
        ----------
        alpha : torch.Tensor
            Веса классов размером [2] для [класс_0, класс_1].
            Пример: torch.tensor([0.75, 0.25])
        gamma : float
            Параметр фокусировки. По умолчанию 2.0.
        reduction : str
            'mean', 'sum' или 'none'.
        """
        super().__init__()
        # Гарантируем, что alpha — это тензор с плавающей точкой
        self.alpha = alpha.float()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        inputs : torch.Tensor
            Логиты модели (без сигмоиды), форма [Batch, ...]
        targets : torch.Tensor
            Метки классов (0 или 1), форма [Batch, ...]
        """
        assert inputs.shape == targets.shape, f"Shape mismatch: {inputs.shape} vs {targets.shape}"
        
        # Приводим targets к long для gather и float для BCE
        targets_long = targets.long()
        targets_float = targets.float()

        # 1. Бинарная кросс-энтропия без редукции
        bce_loss = F.binary_cross_entropy_with_logits(
            inputs, targets_float, reduction="none"
        )

        # 2. Вероятность правильного класса (p_t)
        p_t = torch.exp(-bce_loss)

        # 3. Выбор веса alpha для каждого пикселя/элемента
        # Перемещаем alpha на то же устройство, что и данные
        alpha = self.alpha.to(inputs.device)
        
        # gather выбирает вес: если target=0 -> alpha[0], если target=1 -> alpha[1]
        alpha_t = alpha.gather(0, targets_long.view(-1)).view_as(targets)

        # 4. Фокусирующий коэффициент (1 - p_t)^gamma
        focal_weight = (1 - p_t) ** self.gamma

        # 5. Итоговый лосс
        loss = alpha_t * focal_weight * bce_loss

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        else:
            return loss

In [ ]:
SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)

def compute_metrics(predictions, labels, threshold=0.5):
    preds_bin = (predictions > threshold).astype(int)
    return {
        "accuracy": accuracy_score(labels, preds_bin),
        "f1": f1_score(labels, preds_bin),
        "precision": precision_score(labels, preds_bin),
        "recall": recall_score(labels, preds_bin),
    }

def evaluate_model(model, data_loader, device, loss_func, threshold=0.5):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Validation", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids, attention_mask)
            loss = loss_func(logits, labels)

            probs = torch.sigmoid(logits)

            total_loss += loss.item()
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    all_probs = np.array(all_probs).ravel()
    all_labels = np.array(all_labels).ravel()

    metrics = compute_metrics(all_probs, all_labels, threshold=threshold)
    metrics["loss"] = avg_loss
    return metrics

def train_model(model, train_loader, val_loader, device, num_epochs=3, lr=3e-5, threshold=0.5):
    model.to(device)

    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(0.05 * total_steps)

    loss_func = FocalLoss(
        alpha=torch.tensor([0.7, 1.0], device=device),
        gamma=2.5
    )

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    history = []
    best_val_f1 = -1.0
    best_model_path = os.path.join(SAVE_DIR, "best_model.pt")

    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0.0
        train_probs = []
        train_labels = []

        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
        for batch in loop:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()

            logits = model(input_ids, attention_mask)
            loss = loss_func(logits, labels)

            loss.backward()
            optimizer.step()
            scheduler.step()

            probs = torch.sigmoid(logits)

            total_train_loss += loss.item()
            train_probs.extend(probs.detach().cpu().numpy())
            train_labels.extend(labels.detach().cpu().numpy())

        train_probs = np.array(train_probs).ravel()
        train_labels = np.array(train_labels).ravel()
        train_metrics = compute_metrics(train_probs, train_labels, threshold=threshold)
        train_metrics["loss"] = total_train_loss / len(train_loader)

        val_metrics = evaluate_model(
            model=model,
            data_loader=val_loader,
            device=device,
            loss_func=loss_func,
            threshold=threshold
        )

        current_lr = scheduler.get_last_lr()[0]

        row = {
            "epoch": epoch + 1,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_f1": train_metrics["f1"],
            "train_precision": train_metrics["precision"],
            "train_recall": train_metrics["recall"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_f1": val_metrics["f1"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "lr": current_lr,
        }
        history.append(row)

        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(
            f"train_loss={row['train_loss']:.4f} | train_f1={row['train_f1']:.4f} | "
            f"val_loss={row['val_loss']:.4f} | val_f1={row['val_f1']:.4f}"
        )

        if row["val_f1"] > best_val_f1:
            best_val_f1 = row["val_f1"]
            torch.save(model.state_dict(), best_model_path)
            print(f"Best model saved to: {best_model_path}")

    history_df = pd.DataFrame(history)
    history_csv_path = os.path.join(SAVE_DIR, "training_history.csv")
    history_df.to_csv(history_csv_path, index=False)

    history_json_path = os.path.join(SAVE_DIR, "training_history.json")
    history_df.to_json(history_json_path, orient="records", force_ascii=False, indent=2)

    print(f"History saved to: {history_csv_path}")
    print(f"History saved to: {history_json_path}")

    return model, history_df, best_model_path

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = E5Classifier(model_name=MODEL_NAME)

final_model, history_df, best_model_path = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    threshold=0.5
)

In [ ]:
# loss
plt.figure(figsize=(10, 6))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="s", label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train / Val Loss")
plt.grid(True)
plt.legend()
plt.savefig(os.path.join(SAVE_DIR, "loss_curve.png"), dpi=150, bbox_inches="tight")
plt.show()

# f1
plt.figure(figsize=(10, 6))
plt.plot(history_df["epoch"], history_df["train_f1"], marker="o", label="train_f1")
plt.plot(history_df["epoch"], history_df["val_f1"], marker="s", label="val_f1")
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.title("Train / Val F1")
plt.grid(True)
plt.legend()
plt.savefig(os.path.join(SAVE_DIR, "f1_curve.png"), dpi=150, bbox_inches="tight")
plt.show()

# accuracy
plt.figure(figsize=(10, 6))
plt.plot(history_df["epoch"], history_df["train_accuracy"], marker="o", label="train_accuracy")
plt.plot(history_df["epoch"], history_df["val_accuracy"], marker="s", label="val_accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Train / Val Accuracy")
plt.grid(True)
plt.legend()
plt.savefig(os.path.join(SAVE_DIR, "accuracy_curve.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
best_model = E5Classifier(model_name=MODEL_NAME)
best_model.load_state_dict(torch.load(best_model_path, map_location=device))
best_model.to(device)
best_model.eval()

checkpoint = {
    "model_state_dict": best_model.state_dict(),
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "threshold": 0.5,
}

torch.save(checkpoint, "/kaggle/working/final_checkpoint.pt")
print("Saved: /kaggle/working/final_checkpoint.pt")

In [ ]:
torch.save(model.state_dict(), "best_model.pt")

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH
}, "final_checkpoint_all_save.pt")

In [ ]:
# Загружаем лучшую модель для финального тестирования
model.load_state_dict(torch.load('/kaggle/working/best_model.pt', map_location=device))

model.to(device).eval()

final_predictions = []
final_labels = []

# Сбор предсказаний на тестовом наборе
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(input_ids, attention_mask)
        predictions = torch.sigmoid(outputs).cpu().numpy()
        
        final_predictions.extend(predictions)
        final_labels.extend(labels.cpu().numpy())

# Конвертация в массивы
final_predictions = np.array(final_predictions).ravel()  # Важно: flatten для бинарной задачи
final_labels = np.array(final_labels).ravel()

In [ ]:
# --- Precision-Recall vs Threshold ---
# Расчет метрик для разных порогов
precisions, recalls, thresholds = precision_recall_curve(final_labels, final_predictions)

# Построение графика зависимости от порога
plt.figure(figsize=(10, 6))
plt.plot(thresholds, precisions[:-1], label='Precision', linewidth=2, color='blue')
plt.plot(thresholds, recalls[:-1], label='Recall', linewidth=2, color='orange')
plt.xlabel('Threshold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Precision and Recall vs Decision Threshold', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Бинаризация предсказаний для основных метрик
threshold = 0.5
final_predictions_binary = (final_predictions > threshold).astype(int)

# --- Classification Report ---
print('\n' + '='*50)
print(f'CLASSIFICATION REPORT (threshold={threshold})')
print('='*50)
print(classification_report(final_labels, final_predictions_binary, digits=4))

# --- Confusion Matrix ---
print('\n' + '='*50)
print(f'CONFUSION MATRIX (threshold={threshold})')
print('='*50)
cm = confusion_matrix(final_labels, final_predictions_binary)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'], 
            yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Добавляем столбцы в датафрейм
df_test['model_score'] = final_predictions
df_test['predicted_label'] = (final_predictions > 0.5).astype(int)

# Выводим примеры FP (ложные срабатывания)
fp_df = df_test[(df_test['predicted_label'] == 1) & (df_test['labels'] == 0)]
print(f"\nFalse Positives (FP): {len(fp_df)}")
print("="*80)
for idx, row in fp_df.sample(min(5, len(fp_df))).iterrows():
    print(f"СКОР от модели: {row['model_score']:.4f}")
    print(f"ТЕКСТ: {row['text'][:200]}...")
    print("-"*80)

In [ ]:
# Выводим примеры FN (пропущенные цели)
fn_df = df_test[(df_test['predicted_label'] == 0) & (df_test['labels'] == 1)]
print(f"\nFalse Negatives (FN): {len(fn_df)}")
print("="*80)
for idx, row in fn_df.sample(min(5, len(fn_df))).iterrows():
    print(f"СКОР от модели: {row['model_score']:.4f}")
    print(f"ТЕКСТ: {row['text'][:200]}...")
    print("-"*80)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Распределение для FP
if len(fp_df) > 0:
    axes[0].hist(fp_df['model_score'], bins=20, alpha=0.7, color='red', edgecolor='black')
    axes[0].axvline(x=0.5, color='blue', linestyle='--', label='Threshold=0.5')
    axes[0].set_xlabel('Скор модели (вероятность Positive)')
    axes[0].set_ylabel('Количество')
    axes[0].set_title(f'Распределение скоров FP (n={len(fp_df)})')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    

# Распределение для FN
if len(fn_df) > 0:
    axes[1].hist(fn_df['model_score'], bins=20, alpha=0.7, color='orange', edgecolor='black')
    axes[1].axvline(x=0.5, color='blue', linestyle='--', label='Threshold=0.5')
    axes[1].set_xlabel('Скор модели (вероятность Positive)')
    axes[1].set_ylabel('Количество')
    axes[1].set_title(f'Распределение скоров FN (n={len(fn_df)})')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
plt.tight_layout()
plt.savefig('distribution_plot.png', dpi=300, bbox_inches='tight')  # Сохраняем перед show()
plt.show()

In [ ]:
# Выводим примеры FN (пропущенные цели)
fn_df = df_test[(df_test['predicted_label'] == 0) & (df_test['labels'] == 1) & (df_test['model_score'] < 0.125)]
print(f"\nFalse Negatives (FN): {len(fn_df)}")
print("="*80)
for idx, row in fn_df.sample(min(3, len(fn_df))).iterrows():
    print(f"СКОР от модели: {row['model_score']:.4f}")
    print(f"ТЕКСТ: {row['text'][:200]}...")
    print("-"*80)

In [ ]:
# Функция для предсказания на новых данных
def predict_texts(model, tokenizer, texts, device):
    model.eval()
    predictions = []
    
    for text in texts:
        encoding = tokenizer(
            str(text),
            truncation=True,
            padding='max_length',
            max_length=MAX_LENGTH,
            return_tensors='pt'
        )
        
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)
        
        with torch.no_grad():
            output = model(input_ids, attention_mask)
            prob = torch.sigmoid(output).cpu().item()
            predictions.append(prob)
    
    return predictions

In [ ]:
test_texts = [
    "Фильм оставил очень странное впечатление, с одной стороны визуально он выглядит потрясающе, но сюжет кажется затянутым и местами нелогичным. Актерская игра хорошая, но персонажи раскрыты не до конца, из-за чего сложно сопереживать.",
    
    "Отличная картина с сильной драматургией и глубокими персонажами. Особенно понравилась операторская работа и саундтрек, который идеально дополняет атмосферу. Финал оказался неожиданным и заставил задуматься.",
    
    "Это типичный развлекательный блокбастер, который не пытается быть чем-то большим. Много экшена, спецэффекты на уровне, но сюжет довольно шаблонный и предсказуемый. Смотреть можно, но без ожиданий.",
    
    "Фильм оказался намного лучше, чем я ожидал. История развивается плавно, персонажи живые и убедительные. Особенно понравился главный герой, его мотивация и внутренний конфликт показаны очень качественно.",
    
    "Слишком медленное повествование и перегруженность диалогами делают фильм скучным. Несмотря на интересную идею, реализация подкачала. Несколько сцен можно было бы спокойно вырезать без потери смысла.",
    
    "Очень атмосферное кино с красивой картинкой и отличной музыкой. Сюжет не самый оригинальный, но подача делает его увлекательным. После просмотра остается приятное послевкусие.",
    
    "Актерская игра на высшем уровне, особенно второстепенные персонажи получились яркими и запоминающимися. Однако сценарий местами проседает, и некоторые сюжетные линии выглядят недоработанными.",
    
    "Фильм удивил своей глубиной и философскими подтекстами. Он не для всех, требует внимательного просмотра, но если вникнуть, можно получить настоящее удовольствие и пищу для размышлений.",
    
    "Фильм шедевр! Меня удивил",

    "Унылая работа, не ожидал, что такую интересную и захватывающую историю, можно так мутно изложить"
]
predict_texts(model=model, tokenizer=tokenizer, texts=test_texts, device=device)

In [ ]:
class ModelWithReshape(nn.Module):
    def __init__(self, original_model):
        super().__init__()
        self.original_model = original_model

    def forward(self, input_ids, attention_mask):
        output = self.original_model(input_ids, attention_mask)   # [batch]
        return output.unsqueeze(-1)  # [batch, 1]

In [ ]:
def convert_to_onnx(model, tokenizer, device, output_path="/kaggle/working/model.onnx", max_length=256):
    model.eval()
    wrapped_model = ModelWithReshape(model).to(device)
    wrapped_model.eval()

    dummy_text = "dummy text for onnx export"
    dummy_inputs = tokenizer(
        dummy_text,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )

    dummy_input_ids = dummy_inputs["input_ids"].to(device)
    dummy_attention_mask = dummy_inputs["attention_mask"].to(device)

    torch.onnx.export(
        wrapped_model,
        (dummy_input_ids, dummy_attention_mask),
        output_path,
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=["input_ids", "attention_mask"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch_size", 1: "seq_len"},
            "attention_mask": {0: "batch_size", 1: "seq_len"},
            "logits": {0: "batch_size"}
        }
    )

    print(f"ONNX model saved to: {output_path}")

In [ ]:
convert_to_onnx(model, tokenizer, device, "model.onnx", MAX_LENGTH)